# Arbeitspaket (AP) 4: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung, Schlüssel &amp; Datenqualität</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [6]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /Users/m/projects/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP04


### Datenbankverbindung zu PostgreSQL

In [7]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [8]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Analyst ärgert sich über die vielen JOINs in seinen Auswertungen. Er schlägt vor, in der Tabelle `tickets` zusätzlich die Spalten `cinemaid` und `movie_title` zu speichern:

*«Dann kann ich Tickets pro Kino oder pro Film direkt aus `tickets` zählen.»*

Ist das aus Sicht der Datenmodellierung sinnvoll? Begründen Sie Ihre Antwort und nennen Sie ein **konkretes** Problem, das mit diesem Vorschlag in `cinema.db` entstehen kann.

**Ihre Antwort:**

> ticketid ist über screeningid mit cinemaid verbunden. würde cinemaid auch noch in tickets gespeichert, könnte es zu konflikten kommen. 

### A2 (4 Punkte)

Bei einer Stichprobe in `cinema.db` fällt folgendes Ticket auf:

```
ticketid   purchase_datetime     screeningid   screening_datetime    (aus screenings)
T00003     2025-06-17 01:17:04   ...           2024-09-14 13:00:00
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (accurate, consistent, complete, reliable). Wie würden Sie vorgehen, um herauszufinden, ob es sich um einen Einzelfall handelt?

**Ihre Antwort:**

> Falls mit ... ein leeres Datenfeld genmeint ist, würde ich es als incomplete bezeichnen, wenn ein user manuell ... eingegeben hat eher incosistent. wenn die screeningid einem definierten schema folgt, würde ich nach allem suchen, das nciht dem schema folgt, ansonsten würde ich nach '...' suchen.

### A3 (4 Punkte)

Ein Kollege möchte die Spalte `ticketid` entfernen und stattdessen `seat_number` (z.B. `A1`, `C12`) als **Primärschlüssel** der Tabelle `tickets` verwenden:

*«Jeder Sitzplatz hat eine eindeutige Nummer — das reicht als Schlüssel.»*

Ist `seat_number` als Primärschlüssel geeignet? Begründen Sie mit den Daten in `cinema.db` und schlagen Sie eine Alternative vor, die ohne `ticketid` auskommen würde.

**Ihre Antwort:**

> Furchtbare Idee! Die Plätze werden wiederverwendet und sind somit nciht eindeutig. Ausserdem könnte es auch zu wechsel der seat_number nach dem Kauf kommen.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets (`ticket_count`) und **tatsächlich erzielter Umsatz** (`revenue`, Summe der bezahlten Preise in CHF, gerundet auf 2 Dezimalstellen) pro Kino, absteigend sortiert nach `revenue`. Kinos ohne Tickets müssen nicht erscheinen.

Die SQL-Abfrage wurde aus einem bestehenden Personal-Report kopiert und angepasst:

```sql
SELECT c.name,
       COUNT(*) AS ticket_count,
       ROUND(CAST(SUM(s.base_price) AS NUMERIC), 2) AS revenue
FROM cinemas c
JOIN screenings s ON c.cinemaid    = s.cinemaid
JOIN tickets t    ON s.screeningid = t.screeningid
JOIN staff st     ON c.cinemaid    = st.cinemaid
GROUP BY c.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis der SQL-Abfrage:

```
                name  ticket_count  revenue
0              Kinok          3969  67171.5
1          Cineplexx          3064  49948.0
2  Arthouse Metropol          2832  47780.0
3           Rex Kino          1544  25010.0
4      Pathé Küchlin           436   7181.5
```

Die Tabelle `tickets` enthält insgesamt **2000 Tickets**. Die Controllerin wundert sich: *«Allein Kinok soll fast 4000 Tickets verkauft haben?»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage und erklären Sie, warum ausgerechnet die Ticketzahl von «Pathé Küchlin» bereits stimmt.

**Begründung der Änderungen:**

> Tabelle staff entfernt, enthält keine relevanten Daten. Vermutlich hat das zu den vielen counts geführt.

> zu zählende Spalte in count() eufgeführt um Fehler zu vermeiden.

**Korrigierte SQL-Abfrage:**

In [19]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
        SELECT c.name,
            COUNT(t.ticketid) AS ticket_count,
            ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
        FROM cinemas c
        JOIN screenings s ON c.cinemaid    = s.cinemaid
        JOIN tickets t    ON s.screeningid = t.screeningid
        GROUP BY c.name
        ORDER BY revenue DESC;
""", con=engine)

engine.dispose()
df_B1

,name,ticket_count,revenue
0,Kinok,441,6975.89
1,Pathé Küchlin,436,6731.89
2,Rex Kino,386,5858.15
3,Cineplexx,383,5855.13
4,Arthouse Metropol,354,5595.37


### B2 (8 Punkte)

**Ziel:** Für ein Treueprogramm sollen die **5 umsatzstärksten Kundinnen und Kunden** angeschrieben werden. Ausgegeben werden sollen `customerid`, `firstname`, `lastname`, die Anzahl gekaufter Tickets (`ticket_count`) und der Umsatz (`revenue`, Summe `paid_price`, gerundet auf 2 Dezimalstellen), absteigend sortiert nach `revenue`.

```sql
SELECT c.firstname,
       c.lastname,
       COUNT(t.ticketid) AS ticket_count,
       ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
GROUP BY c.firstname, c.lastname
ORDER BY revenue DESC
LIMIT 5;
```

Tatsächliches Ergebnis:

```
  firstname lastname  ticket_count  revenue
0       Lea     Wolf            33   501.81
1       Jan  Richter            32   462.60
2       Lea    Weber            24   385.02
3      Liam   Krüger            24   375.37
4     Marie    Meier            21   340.24
```

Das Marketing ist irritiert: *«Laut Kassensystem hat keine einzelne Person mehr als 12 Tickets gekauft.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> Customerid der Ausgabe hinzugefügt. Vermutlich gab es mehrere Personen mit den gleichen Namen.

**Korrigierte SQL-Abfrage:**

In [29]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
        SELECT c.customerid,
            c.firstname,
            c.lastname,
            COUNT(t.ticketid) AS ticket_count,
            ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
        FROM customers c
        JOIN tickets t ON c.customerid = t.customerid
        GROUP BY c.customerid, c.firstname, c.lastname
        ORDER BY revenue DESC
        LIMIT 5;    

""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname,ticket_count,revenue
0,CU0460,Lena,Meier,12,190.97
1,CU0108,Jan,Hartmann,11,182.52
2,CU0421,Marie,Meier,10,175.49
3,CU0053,Paul,Braun,10,163.80
4,CU0461,Sara,Keller,10,161.15


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Prüfen Sie die Plausibilitätsregel aus A2: Wie viele Tickets wurden **nach** dem Beginn der zugehörigen Vorstellung gekauft (`purchase_datetime` später als `screening_datetime`)? Geben Sie pro Kino den Namen (`name`) und die Anzahl betroffener Tickets (`invalid_tickets`) aus, absteigend sortiert nach `invalid_tickets`. Begründen Sie Ihre Filterbedingung und warum der Vergleich der beiden Zeitstempel hier korrekt funktioniert.

**Begründung der Vorgehensweise:**

> Zeitstempel funktioniert, weil als datetime formaitierte Spalte.

**SQL-Abfrage:**

In [64]:
engine = create_engine(db_connection_url)
pd.read_sql_query("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE column_name LIKE '%%datetime'
""", con=engine)

,table_name,column_name,data_type
0,screenings,screening_datetime,text
1,tickets,purchase_datetime,text


In [42]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
        SELECT c.name,
            count(t.ticketid) AS invalid_tickets
        FROM cinemas c
        JOIN screenings s ON c.cinemaid = s.cinemaid
        JOIN tickets t ON s.screeningid = t.screeningid
        WHERE s.screening_datetime < t.purchase_datetime
        GROUP BY c.cinemaid, c.name
        ORDER BY invalid_tickets DESC

""", con=engine)

engine.dispose()
df_C1

,name,invalid_tickets
0,Rex Kino,209
1,Pathé Küchlin,201
2,Arthouse Metropol,188
3,Cineplexx,183
4,Kinok,174


### C2 (4 Punkte)

Welche Filmgenres bringen am meisten Umsatz? Geben Sie `genre`, die Anzahl verkaufter Tickets (`ticket_count`) und den Umsatz (`revenue`, Summe `paid_price`, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`) aus, absteigend sortiert nach `revenue`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [56]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
        SELECT m.genre,
            COUNT(t.ticketid) AS ticket_count,
            ROUND(SUM(CAST(t.paid_price AS NUMERIC)), 2) AS revenue
        FROM movies m
        LEFT JOIN screenings s ON m.movieid = s.movieid
        LEFT JOIN tickets t ON s.screeningid = t.screeningid
        GROUP BY m.genre
        ORDER BY revenue DESC

""", con=engine)

engine.dispose()
df_C2

,genre,ticket_count,revenue
0,Sci-Fi,517,7764.75
1,Drama,455,7025.04
2,Comedy,284,4148.48
3,Thriller,233,3677.02
4,Crime,154,2513.62
5,Action,152,2355.51
6,Romance,130,2219.21
7,Animation,75,1312.80


### C3 (4 Punkte)

Das Programmteam möchte schwach nachgefragte Filme aus dem Programm nehmen. Welche Filme haben **weniger als 120 Tickets** verkauft? Filme **ohne** verkaufte Tickets sollen ebenfalls erscheinen (mit `0`). Geben Sie `title` und `ticket_count` aus, aufsteigend sortiert nach `ticket_count`. Begründen Sie Ihre Wahl des JOIN-Typs und warum die Bedingung `< 120` nicht im `WHERE` stehen kann.

**Begründung der Vorgehensweise:**

> LEFT JOINS damit auch Filme ohne verkaufte Tickets erscheinen.

> HAVING: Weil die Bedingung auf das Resultat aus GROUP BY angewendet wird.

**SQL-Abfrage:**

In [66]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
        SELECT m.title,
            COUNT(t.ticketid) AS ticket_count
        FROM movies m
        LEFT JOIN screenings s ON m.movieid = s.movieid
        LEFT JOIN tickets t ON s.screeningid = t.screeningid
        GROUP BY m.movieid, m.title
        HAVING COUNT(t.ticketid) < 120
        ORDER BY ticket_count ASC
""", con=engine)

engine.dispose()
df_C3

,title,ticket_count
0,Spirited Away,75
1,Joker,89
2,The Matrix,98
3,Inception,105


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')